# 00 - Verify raw sources

Checks every file in `data/raw/` against the claims made in the project plan PDF.
Nothing here is taken on trust: each identifier is re-fetched or re-derived from the
downloaded file itself.

Run top to bottom. Every check prints `PASS` or `FAIL`.

In [1]:
import json
import pathlib

import pandas as pd

RAW = pathlib.Path("../data/raw")
assert RAW.is_dir(), f"not found: {RAW.resolve()}"

results = []


def check(label, got, expected):
    ok = got == expected
    results.append(
        {"check": label, "expected": expected, "got": got, "status": "PASS" if ok else "FAIL"}
    )
    print(f"{'PASS' if ok else 'FAIL'}  {label}\n      expected={expected!r}\n      got     ={got!r}")
    return ok


sorted(p.name for p in RAW.iterdir())

['1RX2.pdb',
 'AF-P00374-F1-confidence_v6.json',
 'AF-P00374-F1-model_v6.cif',
 'AF-P00374-F1-model_v6.pdb',
 'AF-P00374-F1-predicted_aligned_error_v6.json',
 'AF-P00382-F1-confidence_v6.json',
 'AF-P00382-F1-model_v6.cif',
 'AF-P00382-F1-model_v6.pdb',
 'AF-P00382-F1-predicted_aligned_error_v6.json',
 'AF-P0ABQ4-F1-confidence_v6.json',
 'AF-P0ABQ4-F1-model_v6.cif',
 'AF-P0ABQ4-F1-model_v6.pdb',
 'AF-P0ABQ4-F1-predicted_aligned_error_v6.json',
 'P00374.json',
 'P00382.json',
 'P0ABQ4.json',
 'trimethoprim_CID5578.json',
 'trimethoprim_CID5578.sdf']

## 1. UniProt accessions

The PDF's page 6 table claims human DHFR is `P00382`. Test that.

In [2]:
def uniprot(acc):
    d = json.loads((RAW / f"{acc}.json").read_text())
    genes = d.get("genes") or [{}]
    return {
        "accession": d["primaryAccession"],
        "name": d["proteinDescription"]["recommendedName"]["fullName"]["value"],
        "organism": d["organism"]["scientificName"],
        "gene": genes[0].get("geneName", {}).get("value"),
        "length": d["sequence"]["length"],
        "sequence": d["sequence"]["value"],
    }


entries = {a: uniprot(a) for a in ("P0ABQ4", "P00382", "P00374")}
pd.DataFrame(entries).T.drop(columns="sequence")

,accession,name,organism,gene,length
P0ABQ4,P0ABQ4,Dihydrofolate reductase,Escherichia coli (strain K12),folA,159
P00382,P00382,Dihydrofolate reductase type 1,Escherichia coli,dhfrI,157
P00374,P00374,Dihydrofolate reductase,Homo sapiens,DHFR,187


In [3]:
print("PDF claim: P0ABQ4 = E. coli DHFR (folA), 159 residues")
check("P0ABQ4 organism is E. coli", entries["P0ABQ4"]["organism"].startswith("Escherichia coli"), True)
check("P0ABQ4 gene is folA", entries["P0ABQ4"]["gene"], "folA")
check("P0ABQ4 length", entries["P0ABQ4"]["length"], 159)

print("\nPDF claim: P00382 = HUMAN DHFR, 187 residues")
check("P00382 organism is human", entries["P00382"]["organism"], "Homo sapiens")
check("P00382 length", entries["P00382"]["length"], 187)

print("\nWhat P00382 actually is:")
print("   ", entries["P00382"]["name"], "|", entries["P00382"]["organism"],
      "|", entries["P00382"]["length"], "aa | gene", entries["P00382"]["gene"])

print("\nCorrect human DHFR accession:")
check("P00374 organism is human", entries["P00374"]["organism"], "Homo sapiens")
check("P00374 length is 187", entries["P00374"]["length"], 187)

PDF claim: P0ABQ4 = E. coli DHFR (folA), 159 residues
PASS  P0ABQ4 organism is E. coli
      expected=True
      got     =True
PASS  P0ABQ4 gene is folA
      expected='folA'
      got     ='folA'
PASS  P0ABQ4 length
      expected=159
      got     =159

PDF claim: P00382 = HUMAN DHFR, 187 residues
FAIL  P00382 organism is human
      expected='Homo sapiens'
      got     ='Escherichia coli'
FAIL  P00382 length
      expected=187
      got     =157

What P00382 actually is:
    Dihydrofolate reductase type 1 | Escherichia coli | 157 aa | gene dhfrI

Correct human DHFR accession:
PASS  P00374 organism is human
      expected='Homo sapiens'
      got     ='Homo sapiens'
PASS  P00374 length is 187
      expected=187
      got     =187


True

## 2. Sequence identity, human vs E. coli

The PDF states the two enzymes "share only about 26% sequence identity" and that
"roughly 74% of sequence differs". Compute it with a real global alignment rather
than trusting the number.

In [4]:
from Bio import Align
from Bio.Align import substitution_matrices

ec = entries["P0ABQ4"]["sequence"]
hs = entries["P00374"]["sequence"]

aligner = Align.PairwiseAligner()
aligner.substitution_matrix = substitution_matrices.load("BLOSUM62")
aligner.open_gap_score = -11
aligner.extend_gap_score = -1
aligner.mode = "global"

aln = aligner.align(ec, hs)[0]
a, b = aln[0], aln[1]

ident = sum(1 for x, y in zip(a, b) if x == y and x != "-")
aligned_cols = sum(1 for x, y in zip(a, b) if x != "-" and y != "-")

pct_aligned = 100 * ident / aligned_cols
pct_shorter = 100 * ident / min(len(ec), len(hs))

print(f"alignment length      : {len(a)}")
print(f"identical residues    : {ident}")
print(f"identity / aligned col: {pct_aligned:.1f}%")
print(f"identity / shorter seq: {pct_shorter:.1f}%")
print()
print(f"PDF says ~26% identity -> plausible: {20 <= pct_aligned <= 35}")

alignment length      : 188
identical residues    : 51
identity / aligned col: 32.3%
identity / shorter seq: 32.1%

PDF says ~26% identity -> plausible: True


## 3. Structure files parse and have the right residue counts

In [5]:
import warnings

from Bio.PDB import MMCIFParser, PDBParser
from Bio.PDB.PDBExceptions import PDBConstructionWarning

warnings.simplefilter("ignore", PDBConstructionWarning)

pdbp = PDBParser(QUIET=True)
cifp = MMCIFParser(QUIET=True)

rows = []
for f in sorted(RAW.glob("*.pdb")):
    s = pdbp.get_structure(f.stem, f)
    ch = next(iter(s[0]))
    aa = [r for r in ch if r.id[0] == " "]
    het = [r for r in ch if r.id[0] != " " and r.id[0] != "W"]
    rows.append({"file": f.name, "format": "pdb", "chain": ch.id,
                 "residues": len(aa), "hetero": len(het)})

for f in sorted(RAW.glob("*.cif")):
    s = cifp.get_structure(f.stem, f)
    ch = next(iter(s[0]))
    aa = [r for r in ch if r.id[0] == " "]
    rows.append({"file": f.name, "format": "cif", "chain": ch.id,
                 "residues": len(aa), "hetero": 0})

pd.DataFrame(rows)

,file,format,chain,residues,hetero
0,1RX2.pdb,pdb,A,159,4
1,AF-P00374-F1-model_v6.pdb,pdb,A,187,0
2,AF-P00382-F1-model_v6.pdb,pdb,A,157,0
3,AF-P0ABQ4-F1-model_v6.pdb,pdb,A,159,0
4,AF-P00374-F1-model_v6.cif,cif,A,187,0
5,AF-P00382-F1-model_v6.cif,cif,A,157,0
6,AF-P0ABQ4-F1-model_v6.cif,cif,A,159,0


In [6]:
counts = {r["file"]: r["residues"] for r in rows}
check("AlphaFold P0ABQ4 has 159 residues", counts["AF-P0ABQ4-F1-model_v6.pdb"], 159)
check("AlphaFold P00374 has 187 residues", counts["AF-P00374-F1-model_v6.pdb"], 187)
check("AlphaFold P00382 has 157 residues", counts["AF-P00382-F1-model_v6.pdb"], 157)
check("PDB and mmCIF agree for P0ABQ4",
      counts["AF-P0ABQ4-F1-model_v6.pdb"] == counts["AF-P0ABQ4-F1-model_v6.cif"], True)

PASS  AlphaFold P0ABQ4 has 159 residues
      expected=159
      got     =159
PASS  AlphaFold P00374 has 187 residues
      expected=187
      got     =187
PASS  AlphaFold P00382 has 157 residues
      expected=157
      got     =157
PASS  PDB and mmCIF agree for P0ABQ4
      expected=True
      got     =True


True

## 4. 1RX2 - numbering, Asp27, and what is actually in the file

Phase 2 superposes onto 1RX2 and Phase 5 tests for an Asp27 contact, so the
crystal numbering must line up with UniProt numbering.

In [7]:
from Bio.PDB.Polypeptide import protein_letters_3to1 as three_to_one

s = pdbp.get_structure("1RX2", RAW / "1RX2.pdb")
chain = s[0]["A"]

obs = {r.id[1]: three_to_one.get(r.get_resname().capitalize(), three_to_one.get(r.get_resname(), "X"))
       for r in chain if r.id[0] == " "}

print(f"chain A residues {min(obs)}-{max(obs)}  (n={len(obs)})")
check("1RX2 residue 27 is Asp", obs.get(27), "D")
check("1RX2 residue 20 is Met", obs.get(20), "M")

diffs = [(n, ec[n - 1], obs[n]) for n in sorted(obs) if n <= len(ec) and ec[n - 1] != obs[n]]
print(f"\nsequence differences vs UniProt P0ABQ4: {diffs}")

hets = [(r.get_resname(), r.id[1]) for r in chain if r.id[0].startswith("H_")]
print(f"hetero groups in chain A: {hets}")

chain A residues 1-159  (n=159)
PASS  1RX2 residue 27 is Asp
      expected='D'
      got     ='D'
PASS  1RX2 residue 20 is Met
      expected='M'
      got     ='M'

sequence differences vs UniProt P0ABQ4: [(37, 'N', 'D')]
hetero groups in chain A: [('MN', 300), ('FOL', 161), ('BME', 162), ('NAP', 164)]


## 5. Cited PDB entries - live check against RCSB

Every entry named in the PDF, with its real organism and real bound ligands.

In [8]:
import urllib.request


def rcsb(path):
    with urllib.request.urlopen(f"https://data.rcsb.org/rest/v1/core/{path}", timeout=30) as r:
        return json.load(r)


PDF_CLAIMS = {
    "1RX2": "folate + NADP+",
    "7DFR": "folate + NADP+",
    "1U72": "methotrexate + NADP+",
    "2W3A": "trimethoprim + NADPH",
    "2W9G": "trimethoprim + NADPH",
    "2W9H": "trimethoprim + NADPH",
}

rows = []
for pid, claim in PDF_CLAIMS.items():
    entry = rcsb(f"entry/{pid}")
    org = rcsb(f"polymer_entity/{pid}/1")["rcsb_entity_source_organism"][0]["scientific_name"]
    ligs = []
    for eid in entry["rcsb_entry_container_identifiers"].get("non_polymer_entity_ids", []) or []:
        np_ = rcsb(f"nonpolymer_entity/{pid}/{eid}")["pdbx_entity_nonpoly"]
        ligs.append(np_["comp_id"])
    rows.append({"pdb": pid, "organism": org.title(), "ligands": ",".join(sorted(ligs)),
                 "pdf_says": claim})

ligand_table = pd.DataFrame(rows)
ligand_table

,pdb,organism,ligands,pdf_says
0,1RX2,Escherichia Coli,"BME,FOL,MN,NAP",folate + NADP+
1,7DFR,Escherichia Coli,"FOL,NAP",folate + NADP+
2,1U72,Homo Sapiens,"MTX,NDP",methotrexate + NADP+
3,2W3A,Homo Sapiens,"GOL,NDP,SO4,TOP",trimethoprim + NADPH
4,2W9G,Staphylococcus Aureus,"NDP,TOP",trimethoprim + NADPH
5,2W9H,Staphylococcus Aureus,"EDO,TOP",trimethoprim + NADPH


In [9]:
# NAP = NADP+ (oxidised).  NDP = NADPH (reduced).  TOP = trimethoprim.
# MTX = methotrexate.      FOL = folic acid.
lig = dict(zip(ligand_table["pdb"], ligand_table["ligands"]))

print("PDF: 1U72 = methotrexate + NADP+")
check("1U72 contains NADP+ (NAP)", "NAP" in lig["1U72"], True)
print("   -> 1U72 actually carries", lig["1U72"], "(NDP = NADPH, the reduced form)\n")

print("PDF: 2W9G / 2W9H = trimethoprim + NADPH")
check("2W9G contains NADPH (NDP)", "NDP" in lig["2W9G"], True)
check("2W9H contains NADPH (NDP)", "NDP" in lig["2W9H"], True)
print("   -> 2W9H actually carries", lig["2W9H"], "- no cofactor\n")

print("PDF describes 1RX2 as folate + NADP+ only")
check("1RX2 has only FOL + NAP", sorted(lig["1RX2"].split(",")), ["FOL", "NAP"])
print("   -> 1RX2 actually carries", lig["1RX2"], "(MN and BME are crystallisation additives)")

PDF: 1U72 = methotrexate + NADP+
FAIL  1U72 contains NADP+ (NAP)
      expected=True
      got     =False
   -> 1U72 actually carries MTX,NDP (NDP = NADPH, the reduced form)

PDF: 2W9G / 2W9H = trimethoprim + NADPH
PASS  2W9G contains NADPH (NDP)
      expected=True
      got     =True
FAIL  2W9H contains NADPH (NDP)
      expected=True
      got     =False
   -> 2W9H actually carries EDO,TOP - no cofactor

PDF describes 1RX2 as folate + NADP+ only
FAIL  1RX2 has only FOL + NAP
      expected=['FOL', 'NAP']
      got     =['BME', 'FOL', 'MN', 'NAP']
   -> 1RX2 actually carries BME,FOL,MN,NAP (MN and BME are crystallisation additives)


## 6. Trimethoprim - SMILES in the PDF vs PubChem

In [10]:
pc = json.loads((RAW / "trimethoprim_CID5578.json").read_text())["PC_Compounds"][0]

smiles = {}
formula = None
for p in pc["props"]:
    lab = p["urn"]["label"]
    nm = p["urn"].get("name", "")
    if lab == "SMILES":
        smiles[nm] = p["value"]["sval"]
    if lab == "Molecular Formula":
        formula = p["value"]["sval"]

print("CID        :", pc["id"]["id"]["cid"])
print("formula    :", formula)
print("heavy atoms:", len(pc["atoms"]["element"]))
for k, v in smiles.items():
    print(f"SMILES {k:14s}: {v}")

check("PubChem CID is 5578", pc["id"]["id"]["cid"], 5578)
check("formula is C14H18N4O3", formula, "C14H18N4O3")

CID        : 5578
formula    : C14H18N4O3
heavy atoms: 39
SMILES Absolute      : COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
SMILES Connectivity  : COC1=CC(=CC(=C1OC)OC)CC2=CN=C(N=C2N)N
PASS  PubChem CID is 5578
      expected=5578
      got     =5578
PASS  formula is C14H18N4O3
      expected='C14H18N4O3'
      got     ='C14H18N4O3'


True

In [11]:
# The PDF hardcodes this SMILES. Compare it to PubChem's by canonical form.
# RDKit is the proper tool; fall back to a formula/atom-count comparison if absent.
PDF_SMILES = "COc1cc(cc(c1OC)OC)Cc2cnc(nc2N)N"
PUBCHEM_SMILES = smiles.get("Absolute") or smiles.get("Connectivity")

try:
    from rdkit import Chem

    a = Chem.CanonSmiles(PDF_SMILES)
    b = Chem.CanonSmiles(PUBCHEM_SMILES)
    print("PDF canonical     :", a)
    print("PubChem canonical :", b)
    check("PDF SMILES == PubChem SMILES", a, b)
except ImportError:
    print("RDKit not installed - comparing by composition instead.")
    import re
    from collections import Counter

    def heavy_atoms(smi):
        toks = re.findall(r"Br|Cl|[BCNOSPFIbcnosp]", smi)
        return Counter(t.upper() for t in toks)

    pdf_c, pub_c = heavy_atoms(PDF_SMILES), heavy_atoms(PUBCHEM_SMILES)
    print("PDF     :", dict(pdf_c), "total", sum(pdf_c.values()))
    print("PubChem :", dict(pub_c), "total", sum(pub_c.values()))
    check("heavy-atom composition matches", pdf_c, pub_c)
    check("heavy-atom count matches PubChem record",
          sum(pdf_c.values()), len(pc["atoms"]["element"]))

PDF canonical     : COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC
PubChem canonical : COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC
PASS  PDF SMILES == PubChem SMILES
      expected='COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC'
      got     ='COc1cc(Cc2cnc(N)nc2N)cc(OC)c1OC'


## 7. Summary

In [12]:
summary = pd.DataFrame(results)
n_fail = (summary["status"] == "FAIL").sum()
print(f"{len(summary)} checks, {n_fail} FAIL\n")
summary

20 checks, 5 FAIL



,check,expected,got,status
0,P0ABQ4 organism is E. coli,True,True,PASS
1,P0ABQ4 gene is folA,folA,folA,PASS
2,P0ABQ4 length,159,159,PASS
3,P00382 organism is human,Homo sapiens,Escherichia coli,FAIL
4,P00382 length,187,157,FAIL
5,P00374 organism is human,Homo sapiens,Homo sapiens,PASS
6,P00374 length is 187,187,187,PASS
7,AlphaFold P0ABQ4 has 159 residues,159,159,PASS
8,AlphaFold P00374 has 187 residues,187,187,PASS
9,AlphaFold P00382 has 157 residues,157,157,PASS


In [13]:
fails = summary[summary["status"] == "FAIL"]
print("Claims in the project-plan PDF that do not hold:\n")
for _, r in fails.iterrows():
    print(f"  - {r['check']}: expected {r['expected']!r}, got {r['got']!r}")

outdir = pathlib.Path("../results")
outdir.mkdir(exist_ok=True)
summary.to_csv(outdir / "source_verification.csv", index=False)
ligand_table.to_csv(outdir / "pdb_ligand_check.csv", index=False)
print(f"\nwrote {outdir/'source_verification.csv'} and {outdir/'pdb_ligand_check.csv'}")

Claims in the project-plan PDF that do not hold:

  - P00382 organism is human: expected 'Homo sapiens', got 'Escherichia coli'
  - P00382 length: expected 187, got 157
  - 1U72 contains NADP+ (NAP): expected True, got False
  - 2W9H contains NADPH (NDP): expected True, got False
  - 1RX2 has only FOL + NAP: expected ['FOL', 'NAP'], got ['BME', 'FOL', 'MN', 'NAP']

wrote ../results/source_verification.csv and ../results/pdb_ligand_check.csv
